# Data Cleaning

## 数据检查（初步）

检查各个数据集：
1. 是否缺少文本
2. 文本是否为空
3. 是否缺少标签
4. 标签是否合法
5. 文本是否有重

In [53]:
import pandas as pd

from sentiment_analysis.paths import INTERIM_DIR

train_data = pd.read_csv(INTERIM_DIR / "train.csv")
validation_data = pd.read_csv(INTERIM_DIR / "val.csv")
test_data = pd.read_csv(INTERIM_DIR / "test.csv")

splits = {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}

rows = []

for name, data in splits.items():
    text = data["text"]

    rows.append({
        "NAME": name,
        "COUNT": len(data),
        "MISSING TEXT": text.isna().sum(),
        "EMPTY TEXT": text.fillna("").str.strip().eq("").sum(),
        "MISSING LABEL": data["label"].isna().sum(),
        "INVALID LABEL": (~data["label"].isin([0, 1])).sum(),
        "DUPLICATE TEXT": text.duplicated().sum(),
    })

quality_summary = pd.DataFrame(rows)
display(quality_summary)

,NAME,COUNT,MISSING TEXT,EMPTY TEXT,MISSING LABEL,INVALID LABEL,DUPLICATE TEXT
0,train,22500,0,0,0,0,75
1,validation,2500,0,0,0,0,2
2,test,25000,0,0,0,0,199


## 数据检查（进一步）

从上面的检查结果可以看出，该数据的基础质量很高，仅有一些重复的文本。

接下来仅需对这些重复的文本进行处理，注意到，上面的统计仅仅是在划分出来的三个数据集中内部的重复情况。这里还要考虑在整个数据中是否存在重复。除此以外，还应该考虑到，对于重复的文本，其情感标签是否具有一致性。

In [54]:
# 合并数据
combined = pd.concat(
    [
        train_data.assign(source="train"),
        validation_data.assign(source="validation"),
        test_data.assign(source="test")
    ],
    ignore_index= True
)

combined.head()

,text,label,source
0,"""Algie, the Miner"" is one bad and unfunny sile...",0,train
1,This is a complete Hoax...<br /><br />The movi...,0,train
2,"Nifty little episode played mainly for laughs,...",1,train
3,Perhaps the worst thing about Carlos Mencia's ...,0,train
4,"First of all, I firmly believe that Norwegian ...",0,train


In [55]:
# 检查重复文本
duplicates = combined[
    combined.duplicated(subset="text", keep=False)
].sort_values("text")

print("Duplicate rows:", len(duplicates))

display(duplicates.head(10))

Duplicate rows: 824


,text,label,source
27296,"""Go Fish"" garnered Rose Troche rightly or wron...",0,test
35910,"""Go Fish"" garnered Rose Troche rightly or wron...",0,test
31070,"""Three"" is a seriously dumb shipwreck movie. M...",0,test
29190,"""Three"" is a seriously dumb shipwreck movie. M...",0,test
19514,"""Witchery"" might just be the most incoherent a...",0,train
25058,"""Witchery"" might just be the most incoherent a...",0,test
12779,'Dead Letter Office' is a low-budget film abou...,0,train
16490,'Dead Letter Office' is a low-budget film abou...,0,train
28147,(Spoilers)<br /><br />Oh sure it's based on Mo...,0,test
28440,(Spoilers)<br /><br />Oh sure it's based on Mo...,0,test


In [56]:
# 检查重复文本的情感标签是否冲突
label_counts = combined.groupby("text")["label"].nunique()

conflicting_texts = label_counts[label_counts > 1].index

conflicts = combined[
    combined["text"].isin(conflicting_texts)
].sort_values("text")

print("Conflicting texts:", len(conflicting_texts))

display(conflicts)

Conflicting texts: 0


,text,label,source


In [57]:
# 检查跨数据集重复
source_counts = combined.groupby("text")["source"].nunique()

overlapping_texts = source_counts[source_counts > 1].index

overlaps = combined[
    combined["text"].isin(overlapping_texts)
].sort_values("text")

print("Texts appearing in multiple sources:", len(overlapping_texts))

display(overlaps.head(10))

Texts appearing in multiple sources: 142


,text,label,source
25058,"""Witchery"" might just be the most incoherent a...",0,test
19514,"""Witchery"" might just be the most incoherent a...",0,train
23213,(This is a review of the later English release...,0,validation
34564,(This is a review of the later English release...,0,test
27505,* Some spoilers *<br /><br />This movie is som...,0,test
15755,* Some spoilers *<br /><br />This movie is som...,0,train
8110,"***SPOILERS*** ***SPOILERS*** Well, seeing as ...",1,train
46806,"***SPOILERS*** ***SPOILERS*** Well, seeing as ...",1,test
33845,"**SPOILERS** I rented ""Tesis"" (or ""Thesis"" in ...",0,test
10791,"**SPOILERS** I rented ""Tesis"" (or ""Thesis"" in ...",0,train


In [58]:
# 检查重复的情况
overlap_summary = (
    combined.groupby("text")["source"]
    .agg(lambda x: ", ".join(sorted(set(x))))
    .value_counts()
)

display(overlap_summary)

source
test                 24678
train                22299
validation            2463
test, train            107
train, validation       19
test, validation        16
Name: count, dtype: int64

## 数据清理

上面的结果显示出，train validation test 三个数据集中互有重复，这种情况比较麻烦了。

计划对数据做如下处理：
1. 先对未划分前的数据集（即官方划分的 train 和 test 数据集）进行去重
2. 优先保证训练集的数量，即从去重后的 test 数据集里排删除与 train 重复的 text
3. 对处理后的 train 进行划分，得到 train 和 validation

In [59]:
from sentiment_analysis.paths import RAW_DIR, CLEANED_DIR
from sklearn.model_selection import train_test_split

# 加载原始数据集
train_raw = pd.read_csv(RAW_DIR / "train.csv")
test_raw = pd.read_csv(RAW_DIR / "test.csv")

# 对原始数据集去重
train_unique = train_raw.drop_duplicates(subset= "text").reset_index(drop= True)
test_unique = test_raw.drop_duplicates(subset= "text").reset_index(drop= True)

# 去除 test 中与 train 重复的 text 数据
test_clean = test_unique[
    ~test_unique["text"].isin(train_unique["text"])
].reset_index(drop=True)

# 划分训练集与验证集
train_clean, validation_clean = train_test_split(
    train_unique,
    test_size=0.1,
    random_state=42,                    # 固定随机数种子，保证分层结果可复现
    stratify=train_unique["label"],     # 根据标签进行分层抽样
)

train_clean = train_clean.reset_index(drop=True)
validation_clean = validation_clean.reset_index(drop=True)

In [60]:
# 检查数据基本情况
splits = {
    "train": train_clean,
    "validation": validation_clean,
    "test": test_clean,
}

cleaned_rows = []

for name, data in splits.items():
    text = data["text"]

    cleaned_rows.append({
        "NAME": name,
        "COUNT": len(data),
        "MISSING TEXT": text.isna().sum(),
        "EMPTY TEXT": text.fillna("").str.strip().eq("").sum(),
        "MISSING LABEL": data["label"].isna().sum(),
        "INVALID LABEL": (~data["label"].isin([0, 1])).sum(),
        "DUPLICATE TEXT": text.duplicated().sum(),
    })

cleaned_quality_summary = pd.DataFrame(cleaned_rows)
display(cleaned_quality_summary)

,NAME,COUNT,MISSING TEXT,EMPTY TEXT,MISSING LABEL,INVALID LABEL,DUPLICATE TEXT
0,train,22413,0,0,0,0,0
1,validation,2491,0,0,0,0,0
2,test,24678,0,0,0,0,0


In [61]:
# 合并数据
cleaned_combined = pd.concat(
    [
        train_clean.assign(source="train"),
        validation_clean.assign(source="validation"),
        test_clean.assign(source="test")
    ],
    ignore_index= True
)

cleaned_combined.head()

,text,label,source
0,"There were some scary scenes, which I've alway...",0,train
1,Corbin Bernsen's sent letters to four criminal...,1,train
2,So fortunate were we to see this fantastic fil...,1,train
3,This movie shows how savage the troubles reall...,1,train
4,I sought this film out because I'm a new Frain...,1,train


In [62]:
# 检查重复文本
cleaned_duplicates = cleaned_combined[
    cleaned_combined.duplicated(subset="text", keep=False)
].sort_values("text")

print("Duplicate rows:", len(cleaned_duplicates))

display(cleaned_duplicates.head(10))

Duplicate rows: 0


,text,label,source


In [63]:
# 检查重复文本的情感标签是否冲突
cleaned_label_counts = cleaned_combined.groupby("text")["label"].nunique()

cleaned_conflicting_texts = cleaned_label_counts[cleaned_label_counts > 1].index

cleaned_conflicts = cleaned_combined[
    cleaned_combined["text"].isin(cleaned_conflicting_texts)
].sort_values("text")

print("Conflicting texts:", len(cleaned_conflicting_texts))

display(cleaned_conflicts)

Conflicting texts: 0


,text,label,source


In [64]:
# 检查跨数据集重复
cleaned_source_counts = cleaned_combined.groupby("text")["source"].nunique()

cleaned_overlapping_texts = cleaned_source_counts[cleaned_source_counts > 1].index

cleaned_overlaps = cleaned_combined[
    cleaned_combined["text"].isin(cleaned_overlapping_texts)
].sort_values("text")

print("Texts appearing in multiple sources:", len(cleaned_overlapping_texts))

display(cleaned_overlaps.head(10))

Texts appearing in multiple sources: 0


,text,label,source


In [65]:
# 检查重复的情况
cleaned_overlap_summary = (
    cleaned_combined.groupby("text")["source"]
    .agg(lambda x: ", ".join(sorted(set(x))))
    .value_counts()
)

display(cleaned_overlap_summary)

source
test          24678
train         22413
validation     2491
Name: count, dtype: int64

注意到，在前面的出现的文本中有 `<br />` 等 html 标签，它们并无实际语义，但是如果不处理的话，在后续分词时，会将 br 也视为一个 token。因此，应该对其进行处理。

1. 先检查标签的情况
2. 对标签进行清理
3. 再次检查是否清理完成

In [66]:
# 检查文本中标签情况
import re
import pandas as pd

splits = {
    "train": train_clean,
    "validation": validation_clean,
    "test": test_clean,
}

# 检索类似标签的文本
html_pattern = r"</?[a-zA-Z][^>]*>"

texts = pd.concat(
    [data["text"] for data in splits.values()],
    ignore_index=True,
)

tags = texts.str.findall(html_pattern).explode().dropna()

print("Total HTML-like tags:", len(tags))
print("Unique HTML-like tags:", tags.nunique())

display(tags.value_counts())

Total HTML-like tags: 200448
Unique HTML-like tags: 13


text
<br />                                                                       200421
<i>                                                                               8
</i>                                                                              8
<grin>                                                                            2
<hr>                                                                              1
<http://rogerebert.suntimes.com/apps/pbcs.dll/section?category=ANSWERMAN>         1
<sigh>                                                                            1
<SPOILER>                                                                         1
</SPOILER>                                                                        1
<sp?) classic "Romeo & Juliet". Guess I'll have to rent that next.<br />          1
<em>                                                                              1
</em>                                                                  

从上面的结果可以看出，大多数都是换行标签 `<br />`，还有少量的倾斜段落等标签。这里还有一个需要注意的点，`<grin>`、`<sigh>` 这种并不是真正的 html 标签，且带有一定的情感含义，应当予以保留。

In [67]:
def clean_html(text: str) -> str:
    """仅移除标准的 html 标签。"""

    # 将换行标签替换为空格
    text = re.sub(
        r"</?(?:br|hr|p)\s*/?>",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # 移除倾斜加粗等标签
    text = re.sub(
        r"</?(?:i|em)\s*>",
        "",
        text,
        flags=re.IGNORECASE,
    )

    return text


# 清洗数据
for data in splits.values():
    data["text"] = data["text"].apply(clean_html)

In [68]:
# 检查标签残留情况
texts_clean = pd.concat(
    [data["text"] for data in splits.values()],
    ignore_index=True,
)

remaining_tags = (
    texts_clean.str.findall(html_pattern)
    .explode()
    .dropna()
)

print("Remaining HTML-like tags:")
display(remaining_tags.value_counts())

# Check duplicates within each split
print("\nWithin-split duplicates:")

for name, data in splits.items():
    print(f"{name}: {data['text'].duplicated().sum()}")

# Check duplicates across splits
combined_clean = pd.concat(
    [
        data.assign(source=name)
        for name, data in splits.items()
    ],
    ignore_index=True,
)

source_counts = combined_clean.groupby("text")["source"].nunique()

overlapping_texts = source_counts[source_counts > 1]

print("\nTexts appearing in multiple splits:", len(overlapping_texts))

Remaining HTML-like tags:


text
<grin>                                                                       2
<http://rogerebert.suntimes.com/apps/pbcs.dll/section?category=ANSWERMAN>    1
<sigh>                                                                       1
<SPOILER>                                                                    1
</SPOILER>                                                                   1
<sp?) classic "Romeo & Juliet". Guess I'll have to rent that next.  >        1
Name: count, dtype: int64


Within-split duplicates:
train: 0
validation: 0
test: 0

Texts appearing in multiple splits: 0


对清洗后的数据进行检查，可以看到，数据已经完全符合要求。

In [69]:
# 保存数据
CLEANED_DIR.mkdir(parents= True, exist_ok= True)

train_clean.to_csv(CLEANED_DIR / "train.csv", index= False)
validation_clean.to_csv(CLEANED_DIR / "val.csv", index= False)
test_clean.to_csv(CLEANED_DIR / "test.csv", index= False)